In [1]:
# Import Python libraries
# This script was tested with zarr v2.13.6, v2.18.3, and v3.0.5
import zarr
import numpy as np

In [2]:
# Function to open a Zarr file
def open_zarr(path: str) -> zarr.Group:
    store = (
        zarr.storage.ZipStore(path, mode="r") if path.endswith(".zip")
        else zarr.storage.LocalStore(path)
    )
    return zarr.open_group(store=store, mode="r")


In [4]:
# For example, use the above function to open the cells Zarr file, which contains segmentation mask Zarr arrays
root = open_zarr("../data/Xenium_V1_humanLung_Cancer_FFPE_outs/cells.zarr.zip")

In [5]:
# Look at group array info and structure
root.info
root.tree() # shows structure, array dimensions, data types

/
├── cell_id (162254, 2) uint32
├── cell_summary (162254, 8) float64
├── masks
│   ├── 0 (17098, 51187) uint32
│   ├── 1 (17098, 51187) uint32
│   └── homogeneous_transform (4, 4) float32
└── polygon_sets
    ├── 0
    │   ├── cell_index (165099,) uint32
    │   ├── method (165099,) uint32
    │   ├── num_vertices (165099,) int32
    │   └── vertices (165099, 50) float32
    └── 1
        ├── cell_index (162254,) uint32
        ├── method (162254,) uint32
        ├── num_vertices (162254,) int32
        └── vertices (162254, 50) float32

In [7]:
# Create cell and nucleus segmentation mask np array objects to read or modify
cellseg_mask = np.array(root["masks"]["1"])
nucseg_mask = np.array(root["masks"]["0"])

In [9]:
# Show dimensions of the 2D segmentation mask arrays (also shown in .tree())
# .ndim() shows number of dimensions
# The shape should match the number of pixels in the morphology image.
cellseg_mask.shape

(17098, 51187)

In [10]:
nucseg_mask.shape

(17098, 51187)

In [11]:
# Show max value of cells in the masks (value=0 are background pixels)
# The .max() method counts all the values that are not 0, which should equal
# the total cells detected in the dataset (reported in e.g., analysis_summary.html
# summary tab metric).
cellseg_mask.max()
nucseg_mask.max()

np.uint32(165099)

In [12]:
# Examples for exploring file contents
# How to show array
root["masks"]["0"][0:9] # or root["masks/0"]
root["cell_summary"][0:9]

array([[2.06089813e+02, 1.49589819e+03, 6.84568775e+01,            nan,
                   nan,            nan, 0.00000000e+00, 0.00000000e+00],
       [2.01765823e+02, 1.81621082e+03, 4.91300018e+01, 2.01929810e+02,
        1.81627075e+03, 2.12685945e+01, 0.00000000e+00, 1.00000000e+00],
       [1.79024506e+02, 2.16725391e+03, 1.19618911e+02, 1.79057953e+02,
        2.16857129e+03, 7.47787527e+01, 0.00000000e+00, 1.00000000e+00],
       [1.86060654e+02, 2.16330908e+03, 9.42410972e+01, 1.86076065e+02,
        2.16248047e+03, 5.91095334e+01, 0.00000000e+00, 1.00000000e+00],
       [2.00246887e+02, 2.19859351e+03, 1.20341411e+02, 1.98514633e+02,
        2.19835815e+03, 5.24264082e+01, 0.00000000e+00, 1.00000000e+00],
       [1.96527405e+02, 2.27076929e+03, 1.97965007e+02, 1.94776794e+02,
        2.26625269e+03, 5.60840645e+01, 0.00000000e+00, 2.00000000e+00],
       [2.02888168e+02, 2.26642529e+03, 8.86868782e+01, 2.00892975e+02,
        2.26445972e+03, 3.57185950e+01, 0.00000000e+00, 1.

In [13]:
# How to show attribute values
root.attrs["major_version"]
root.attrs["segmentation_methods"]

['Segmented by boundary stain (ATP1A1/CD45/E-Cadherin)',
 'Segmented by interior stain (18S)',
 'Segmented by nucleus expansion of 5.0µm',
 'Segmented by nuclear stain (DAPI)']

In [14]:
# How to list out attribute names and values
dict(root.attrs.items())
dict(root['cell_summary'].attrs.items())

{'column_descriptions': ['Cell centroid in X',
  'Cell centroid in Y',
  'Cell area',
  'Nucleus centroid in X',
  'Nucleus centroid in Y',
  'Nucleus area',
  'z_level',
  'Nucleus count'],
 'column_names': ['cell_centroid_x',
  'cell_centroid_y',
  'cell_area',
  'nucleus_centroid_x',
  'nucleus_centroid_y',
  'nucleus_area',
  'z_level',
  'nucleus_count']}